# Evaluierung

**Name:** Sahar SAFFI


In [2]:
# install packages
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score", "sacrebleu", "pandas"])
print("done")

done


In [3]:
# Imports
import pandas as pd
from rouge_score import rouge_scorer
from sacrebleu.metrics import BLEU
print("Imports valid")

Imports valid


In [9]:
#Load data
model1 = pd.read_csv("/home/jovyan/code/results/results_model1_inference.csv")
print(f"Model 1 geladen: {len(model1)} Zeilen")
print(model1.columns.tolist())

Model 1 geladen: 643 Zeilen
['id', 'answer']


In [10]:
# Load data model 2 and model 3
model2 = pd.read_csv("/home/jovyan/code/results/results_model2_fewshot.csv")
model3 = pd.read_csv("/home/jovyan/code/results/results_model3_rag.csv")
print(f"Model 2 geladen: {len(model2)} Zeilen, Spalten: {model2.columns.tolist()}")
print(f"Model 3 geladen: {len(model3)} Zeilen, Spalten: {model3.columns.tolist()}")


Model 2 geladen: 643 Zeilen, Spalten: ['id', 'answer']
Model 3 geladen: 643 Zeilen, Spalten: ['id', 'prompt', 'response']


In [11]:
# combine all modells
merged2 = model1[["id","answer"]].merge(model2[["id","answer"]], on="id", suffixes=("_ref","_m2")).dropna()
merged3 = model1[["id","answer"]].merge(model3[["id","response"]], on="id").dropna()

print(f"Model 2 compare: {len(merged2)}")
print(f"Model 3 compare: {len(merged3)}")

Model 2 compare: 643
Model 3 compare: 643


In [12]:
# compare ROUGE for all 3 models
from rouge_score import rouge_scorer
from sacrebleu.metrics import BLEU

def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rL = [], [], []
    for pred, ref in zip(predictions, references):
        s = scorer.score(str(ref), str(pred))
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rL.append(s["rougeL"].fmeasure)
    return {
        "ROUGE-1": round(sum(r1)/len(r1), 4),
        "ROUGE-2": round(sum(r2)/len(r2), 4),
        "ROUGE-L": round(sum(rL)/len(rL), 4),
    }

def compute_bleu(predictions, references):
    bleu = BLEU(effective_order=True)
    result = bleu.corpus_score(predictions, [references])
    return {"BLEU": round(result.score/100, 4)}

# Model 2 vs Model 1
ref = merged2["answer_ref"].astype(str).tolist()
m2  = merged2["answer_m2"].astype(str).tolist()
m2_scores = {**compute_rouge(m2, ref), **compute_bleu(m2, ref)}

# Model 3 vs Model 1
ref3 = merged3["answer"].astype(str).tolist()
m3   = merged3["response"].astype(str).tolist()
m3_scores = {**compute_rouge(m3, ref3), **compute_bleu(m3, ref3)}

print("Model 2:", m2_scores)
print("Model 3:", m3_scores)

Model 2: {'ROUGE-1': 0.3794, 'ROUGE-2': 0.1741, 'ROUGE-L': 0.2596, 'BLEU': 0.1116}
Model 3: {'ROUGE-1': 0.0548, 'ROUGE-2': 0.0048, 'ROUGE-L': 0.0438, 'BLEU': 0.0002}


In [13]:
results = {
    "Model 1 - Inference (Gemma 3 27B)":      {"ROUGE-1": "Referenz", "ROUGE-2": "Referenz", "ROUGE-L": "Referenz", "BLEU": "Referenz"},
    "Model 2 - Fine-tuned (flan-t5-base)":     m2_scores,
    "Model 3 - RAG (TF-IDF + flan-t5-base)":  m3_scores,
}

df_results = pd.DataFrame(results).T
print(df_results.to_string())
df_results.to_csv("evaluation_results.csv")
print("\n save: evaluation_results.csv")

                                        ROUGE-1   ROUGE-2   ROUGE-L      BLEU
Model 1 - Inference (Gemma 3 27B)      Referenz  Referenz  Referenz  Referenz
Model 2 - Fine-tuned (flan-t5-base)      0.3794    0.1741    0.2596    0.1116
Model 3 - RAG (TF-IDF + flan-t5-base)    0.0548    0.0048    0.0438    0.0002

 save: evaluation_results.csv
